# Semana 12
## PCA, t-SNE y visualizacion de alta dimensionalidad

**Objetivo**: aplicar reducción de dimensionalidad y exportar una proyección interpretable en Tableau.

**Herramientas teoricas de la semana**
- escalado
- `PCA`
- `t-SNE`
- preservacion global vs local


### Agenda sugerida de 4 horas
- 0:00 - 0:35: teoria de alta dimensionalidad
- 0:35 - 1:20: escalado y `PCA`
- 1:20 - 2:10: `t-SNE` y sensibilidad a hiperparametros
- 2:10 - 3:20: scatter final y exportacion
- 3:20 - 4:00: limites interpretativos y cierre


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from _shared import (
    make_base_sales,
    introduce_quality_issues,
    profile_dataframe,
    clean_sales_data,
    build_star_schema,
    save_for_tableau,
    ensure_output_dir,
    contrast_ratio,
    make_high_dimensional_dataset,
)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
WEEK = "week-12"
OUTPUT_DIR = ensure_output_dir(WEEK)

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

hd = make_high_dimensional_dataset(n_samples=700, n_features=12, centers=4, seed=37)
feature_cols = [c for c in hd.columns if c.startswith('f_')]
X = hd[feature_cols]
X_scaled = StandardScaler().fit_transform(X)


In [ ]:
pca = PCA(n_components=2, random_state=37)
pca_coords = pca.fit_transform(X_scaled)
hd['pca_1'] = pca_coords[:, 0]
hd['pca_2'] = pca_coords[:, 1]
explained = pca.explained_variance_ratio_
explained


In [ ]:
tsne = TSNE(n_components=2, perplexity=30, init='pca', learning_rate='auto', random_state=37)
tsne_coords = tsne.fit_transform(X_scaled)
hd['tsne_1'] = tsne_coords[:, 0]
hd['tsne_2'] = tsne_coords[:, 1]
hd.head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.scatterplot(data=hd, x='pca_1', y='pca_2', hue='cluster', palette='Greys', ax=axes[0], legend=False)
axes[0].set_title(f'PCA (varianza explicada: {explained.sum():.2%})')
sns.scatterplot(data=hd, x='tsne_1', y='tsne_2', hue='cluster', palette='Greys', ax=axes[1], legend=False)
axes[1].set_title('t-SNE (estructura local)')
plt.tight_layout()


In [ ]:
methodology_notes = pd.DataFrame([
    {'item': 'scaling', 'value': 'StandardScaler applied to all numeric features'},
    {'item': 'pca_explained_variance', 'value': float(explained.sum())},
    {'item': 'tsne_perplexity', 'value': 30},
    {'item': 'warning', 'value': 't-SNE preserves local neighborhoods, not global distances'},
])
methodology_notes


In [ ]:
save_for_tableau(hd[['sample_id', 'cluster', 'segment', 'pca_1', 'pca_2', 'tsne_1', 'tsne_2']], WEEK, 'embedding_coordinates')
save_for_tableau(methodology_notes, WEEK, 'methodology_notes')


### Uso teorico de herramientas
- `StandardScaler` operacionaliza la necesidad de comparabilidad entre variables.
- `PCA` y `TSNE` hacen visible la diferencia entre estructura global y local.
- El CSV exportado puede abrirse en Tableau como scatter plot con color por cluster.
